# Laboratório 09: Arquitetura RAG (Blade Runner 2049)
**Objetivo:** Implementar um pipeline RAG de produção utilizando índice hierárquico HNSW, Query Transformation (HyDE) e Re-ranking com Cross-Encoders.

## Passo 0: Preparação do Ambiente
Instalação das bibliotecas de processamento vetorial, modelos locais da Hugging Face e SDK do Gemini para o HyDE.

In [2]:
!pip install -q sentence-transformers faiss-cpu google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 89.9 MB/s eta 0:00:00


## Passo 1: Construcao do Grafo HNSW (Wallace Corp Source)
- Aqui, aplicamos a separação de responsabilidades.
- Cria dataset com 20 fragmentos
- Cria classe `WallaceKnowledgeBase` para encapsular a lógica do FAISS (HNSW) e o Bi-Encoder (`all-MiniLM-L6-v2`)

In [3]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# 20 Fragmentos Técnicos
documentos_tecnicos = [
    "Protocolo Voight-Kampff v2.0: Medição de contrações pupilares e dilatação capilar para detecção de respostas empáticas em Replicantes Nexus-8.",
    "Modelo Nexus-9: Obediência garantida através de condicionamento genético de submissão na medula espinhal, imune a desvios de Baseline.",
    "Teste de Baseline (Pós-Trauma): O oficial deve recitar o poema de Nabokov. Qualquer desvio na cadência verbal indica dissonância cognitiva.",
    "Implantes de Memória Sintética: Fabricados pelo Dr. Ana Stelline. Empregados para estabilizar a psique de modelos Nexus, evitando colapsos emocionais prematuros.",
    "Falha de Hardware em Spinners (Série Peugeot): Perda de altitude repentina geralmente associada ao congelamento do duto de propulsão antigravitacional inferior.",
    "Detecção de Radiação de Las Vegas: Replicantes enviados à zona de quarentena devem tomar suplementos de iodo 400mg a cada ciclo solar.",
    "Sincronização de IA Holográfica (Joi): Atualizações de firmware requerem o Emanador Wallace para persistência de memória fora do console base.",
    "Procedimento de 'Aposentadoria': Oficial Blade Runner está autorizado a utilizar força letal (Blaster LAPD calibre .44) caso o alvo Replicante apresente hostilidade.",
    "Marcador Genético Ocular: Todo Replicante fabricado após o Blecaute possui um número de série microscópico gravado no canto inferior direito do globo ocular.",
    "Síndrome do Blecaute de 2022: Perda de dados digitais históricos devido à detonação EMP. Manuais anteriores a esta data só existem em formato físico (papel).",
    "Instabilidade Emocional Nexus-8: Período de vida aberto causa crises existenciais após o 4º ano de operação contínua sem formatação.",
    "Calibração do Blaster LAPD: Ajuste o emissor de plasma térmico a cada 500 disparos para evitar superaquecimento do tambor magnético.",
    "Emanador Portátil Wallace: Dispositivo de projeção de partículas de luz que permite a entidades de IA como a série Joi interagirem com variações climáticas (chuva, neve).",
    "Anatomia Replicante: Ossos reforçados com fibra de carbono e fluidos biológicos sintéticos tolerantes a variações extremas de temperatura (-50C a +120C).",
    "Cultivo de Proteína (Fazendas Sapper Morton): Processamento de larvas em biotanques de alta pressão para exportação de pacotes de ração nutritiva.",
    "Dissonância de Implante de Memória: O sujeito passa a questionar se suas lembranças de infância são reais ou fabricadas, gerando ansiedade severa e falha no Baseline.",
    "Arquivo de DNA Wallace Corp: Matrizes de genoma guardadas a zero absoluto nas abóbadas da sede; usadas para esculpir a próxima geração de trabalhadores off-world.",
    "Viagens Off-World: O trânsito para as colônias mineiras exige que modelos de combate entrem em estase criogênica durante os 8 meses de voo.",
    "Identificação de Nexus-8: Busca por registros pré-blecaute nos terminais do Departamento de Polícia de Los Angeles (LAPD).",
    "Mercado Negro de San Diego: Tráfico de peças biológicas e upgrades ilegais de memória sintética prospera nas ruínas do distrito do lixo."
]

class WallaceKnowledgeBase:
    def __init__(self, documentos):
        print("Iniciando Bi-Encoder (Modelo de Embedding)...")

        # Bi-Encoder
        self.encoder = SentenceTransformer('all-MiniLM-L6-v2')
        self.documentos = documentos
        self.dimensao = self.encoder.get_sentence_embedding_dimension()

        # Hierarchical Navigable Small World no FAISS
        # M = 32
        self.M = 32
        self.index = faiss.IndexHNSWFlat(self.dimensao, self.M)
        self.index.hnsw.efConstruction = 64
        self.index.hnsw.efSearch = 32

        self._indexar_documentos()

    def _indexar_documentos(self):
        print("Vetorizando manuais e construindo o Grafo HNSW...")
        embeddings = self.encoder.encode(self.documentos, convert_to_numpy=True)

        # Normalização
        faiss.normalize_L2(embeddings)
        self.index.add(embeddings)
        print(f"Grafo pronto! {self.index.ntotal} documentos indexados no FAISS.")

    def retrieve(self, vetor_query, top_k=10):
        distancias, indices = self.index.search(vetor_query, top_k)
        resultados = [self.documentos[i] for i in indices[0]]
        return resultados

# Instanciando Base
db_wallace = WallaceKnowledgeBase(documentos_tecnicos)

Iniciando Bi-Encoder (Modelo de Embedding)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/tmp/ipykernel_7472/209587447.py:35: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  self.dimensao = self.encoder.get_sentence_embedding_dimension()


Vetorizando manuais e construindo o Grafo HNSW...
Grafo pronto! 20 documentos indexados no FAISS.


## Passo 2: Query Transformation (HyDE) e Passo 3: Busca Rápida Bi-Encoder
A classe `HyDETransformer` intercepta a pergunta coloquial do policial e alucina um documento técnico usando o Gemini. Em seguida, usamos esse documento falso para buscar os Top-10 fragmentos reais no grafo HNSW.

In [5]:
import numpy as np
from google import genai
from google.genai import types

class HyDETransformer:
    def __init__(self, api_key):
        self.client = genai.Client(api_key=api_key)

    def transformar_query(self, query_coloquial):
        prompt = f"""
        Você é um perito técnico da Wallace Corporation e da LAPD (Blade Runner).
        Escreva um breve trecho de manual técnico detalhando o seguinte problema coloquial:
        "{query_coloquial}"
        Use jargões corporativos pesados (ex: Teste de Baseline, Nexus-9, Implantes de Memória, etc). Não resolva o problema, apenas o descreva com linguajar altamente técnico.
        Retorne APENAS o texto técnico, sem saudações ou explicações.
        """
        response = self.client.models.generate_content(
            model='gemini-2.5-flash',
            contents=prompt,
            config=types.GenerateContentConfig(temperature=0.3)
        )
        return response.text

# A Query de um policial confuso na rua
query_oficial = "o replicante surtou depois de ver um cavalo de madeira, ele tá chorando e não consegue falar os números direito."
print(f"Query Original do Policial: '{query_oficial}'\n")

# chave API GEMINI AQUI !!!!
MINHA_API_KEY = " "
hyde = HyDETransformer(MINHA_API_KEY)

print("1. Gerando Documento (HyDE)...")
doc_hipotetico = hyde.transformar_query(query_oficial)
print(f"Doc HyDE (Alucinação Técnica):\n{doc_hipotetico}\n")

print("2. Vetorizando Doc HyDE e fazendo busca (Top-10) no HNSW...")
# Vetorizando o documento
vetor_hyde = db_wallace.encoder.encode([doc_hipotetico], convert_to_numpy=True)
faiss.normalize_L2(vetor_hyde)

documentos_recuperados = db_wallace.retrieve(vetor_hyde, top_k=10)

print("=== Top-10 Documentos Recuperados (Bi-Encoder) ===")
for i, doc in enumerate(documentos_recuperados, 1):
    print(f"{i}. {doc}")

Query Original do Policial: 'o replicante surtou depois de ver um cavalo de madeira, ele tá chorando e não consegue falar os números direito.'

1. Gerando Documento Hipotético (HyDE)...
Doc HyDE (Alucinação Técnica):
**Relatório de Incidente: Desvio Comportamental Anômalo em Unidade Nexus-9**

**Referência:** Protocolo de Avaliação Pós-Incidente 7.3.1 - Anomalias de Processamento Sensorial/Cognitivo.

**Descrição do Incidente:**
A Unidade Bioengenheirada Modelo Nexus-9 (N9-XXX), identificada pelo número de série [INSERIR_SN], manifestou uma disfunção cognitiva aguda e desregulação emocional severa. O evento foi precipitado após a exposição visual a um artefato de madeira esculpida, categorizado como representação zoomórfica equina. Imediatamente após a percepção do estímulo, a unidade exibiu lacrimação profusa e vocalizações de angústia, indicativas de um estado de estresse psicogênico não previsto pelos parâmetros de estabilidade psicoemocional do modelo. Concomitantemente, foi observ

## Passo 3: O Filtro Fino (Re-ranking com Cross-Encoder)
A Similaridade de Cosseno (Bi-Encoder) é rápida, mas "burra" semanticamente. Agora usamos um Cross-Encoder profundo. Ele vai ler a query original do policial junto com cada um dos 10 documentos recuperados, palavra por palavra simultaneamente (Atenção Cruzada), para extrair os 3 mais relevantes que realmente salvam a vida do policial.

In [6]:
from sentence_transformers import CrossEncoder

class Reranker:
    def __init__(self):
        print("Iniciando Cross-Encoder (O Funil Fino)...")
        # Modelo de Re-ranking open-source
        self.cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

    def rerank(self, query_original, docs_recuperados, top_k=3):
        # O modelo Cross-Encoder exige pares (Query Original + Documento Candidato)
        pares = [[query_original, doc] for doc in docs_recuperados]

        # Calculando o "Score de Atenção" profundo para cada par
        scores = self.cross_encoder.predict(pares)

        # Juntando os documentos com seus scores e ordena do maior para o menor
        docs_com_scores = list(zip(scores, docs_recuperados))
        docs_ordenados = sorted(docs_com_scores, key=lambda x: x[0], reverse=True)

        return docs_ordenados[:top_k]

# Executando o Re-ranking
reranker = Reranker()
resultados_finais = reranker.rerank(query_oficial, documentos_recuperados, top_k=3)

print("\n=== Top-3 Documentos Finais (Cross-Encoder Re-ranked) ===")
for score, doc in resultados_finais:
    # Mostrando a relevância matemática (score) e o manual definitivo
    print(f"[Score: {score:.4f}] {doc}")

Iniciando Cross-Encoder (O Funil Fino)...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]


=== Top-3 Documentos Finais (Cross-Encoder Re-ranked) ===
[Score: -4.2053] Marcador Genético Ocular: Todo Replicante fabricado após o Blecaute possui um número de série microscópico gravado no canto inferior direito do globo ocular.
[Score: -9.2044] Protocolo Voight-Kampff v2.0: Medição de contrações pupilares e dilatação capilar para detecção de respostas empáticas em Replicantes Nexus-8.
[Score: -10.3387] Instabilidade Emocional Nexus-8: Período de vida aberto causa crises existenciais após o 4º ano de operação contínua sem formatação.
